# AI651 Assignment 3 - Q3 to Q7: LargeST San Diego
Run from the `notebooks/` folder. Code lives in `../src/`; figures go to `../report/figures/`.

**Prerequisite:** the four raw files (`ca_his_raw_2020.h5`, `ca_his_raw_2021.h5`, `ca_meta.csv`, `ca_rn_adj.npy`) are in `../data/raw/`.

In [ ]:
import sys, json, platform, time
sys.path.append("../src")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import data, plots
from train import Config, make_loaders, run_experiment
from model import build_model
from utils import set_seed

SEED = 42
set_seed(SEED)
RAW_DIR, PROCESSED_DIR = "../data/raw", "../data/processed"
RESULTS_DIR, FIGURE_DIR = "../results", "../report/figures"

## Q3(a) Extraction and resampling
Run once. Loading the CA `.h5` files needs several GB of RAM; each year is loaded, cut to the 716 SD sensors, then freed.

In [ ]:
data.build_dataset(RAW_DIR, PROCESSED_DIR)
flow, timestamps, adjacency, meta_sd = data.load_processed(PROCESSED_DIR)
print("flow (T, N):", flow.shape, "| first:", timestamps[0], "| last:", timestamps[-1])
print("adjacency:", adjacency.shape)

In [ ]:
report = data.missing_value_report(flow)
print(json.dumps(report, indent=2))
missing_per_sensor = np.isnan(flow).mean(axis=0)
print("Top 5 sensors by missing fraction:", np.sort(missing_per_sensor)[-5:])

In [ ]:
sensor_indices = [0, 100, 300]          # choose any three
plots.plot_sensors_one_week(flow, timestamps, sensor_indices, meta_sd["ID"].to_numpy(),
                            week_start="2020-03-02",
                            save_path=f"{FIGURE_DIR}/q3a_sensors_week.pdf");

## Q3(b) Graph construction

In [ ]:
summary = data.graph_summary(adjacency)
print(json.dumps(summary, indent=2))
print("Distinct nonzero weight values (first 10):", np.unique(adjacency[adjacency != 0])[:10])

In [ ]:
plots.plot_adjacency_heatmap(adjacency, save_path=f"{FIGURE_DIR}/q3b_adj_heatmap.pdf")
plots.plot_degree_histogram(adjacency, save_path=f"{FIGURE_DIR}/q3b_degree_hist.pdf");

## Q3(c) Neighbourhoods

In [ ]:
selected_sensor = int(np.argmax((adjacency != 0).sum(axis=1)))   # a well-connected sensor; change if you like
one_hop = data.k_hop_neighbours(adjacency, selected_sensor, 1)
two_hop = data.k_hop_neighbours(adjacency, selected_sensor, 2)
print(f"Sensor index {selected_sensor} (ID {meta_sd['ID'][selected_sensor]})")
print("one-hop:", one_hop)
print("two-hop (includes one-hop):", two_hop, f"-> {len(two_hop)} nodes")

## Q4(a) Splits, normalisation, windows

In [ ]:
config = Config(seed=SEED)       # edit hyperparameters here; shared by ALL models
loaders, mean, std, adjacency, splits = make_loaders(config)
print(f"train-only mean={mean:.2f}, std={std:.2f}")
for name in ("train", "val", "test"):
    print(name, "steps:", splits[name].raw_flow.shape[0], "| windows:", len(loaders[name].dataset),
          "| range:", splits[name].timestamps[0], "->", splits[name].timestamps[-1])
history, target, mask = next(iter(loaders["train"]))
print("batch shapes:", tuple(history.shape), tuple(target.shape), tuple(mask.shape))

## Q4(b) Architecture check

In [ ]:
for variant in ("temporal", "fixed", "adaptive", "hybrid"):
    model = build_model(variant, adjacency, data.INPUT_LENGTH, data.HORIZON,
                        config.hidden_dim, config.embedding_dim, config.top_k)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    out = model(history[:2])
    print(f"{variant:9s} params={n_params:>7,d}  output shape={tuple(out.shape)}")

## Q4(c) and Q5: train all four variants
Same encoder, head, splits, seed, optimiser and epoch budget. Slow on CPU: set `train_stride=2` or `max_epochs` lower in `Config` if needed (apply the same value to all four). Results are saved to `../results/`, so you can re-run only one cell.

In [ ]:
prepared = (loaders, mean, std, adjacency, splits)    # reuse loaders for all runs
results = {}
results["temporal"] = run_experiment("temporal", config, prepared)

In [ ]:
results["fixed"] = run_experiment("fixed", config, prepared)

In [ ]:
results["adaptive"] = run_experiment("adaptive", config, prepared)

In [ ]:
results["hybrid"] = run_experiment("hybrid", config, prepared)
print("learned alpha:", results["hybrid"].get("final_alpha"))

## Q5(b) Visualise the learned adjacency

In [ ]:
learned = np.load(f"{RESULTS_DIR}/adaptive_learned_adjacency.npy")
plots.plot_learned_adjacency(learned, save_path=f"{FIGURE_DIR}/q5b_adaptive_adj.pdf")
print("row sums (should be 1):", learned.sum(axis=1)[:3], "| nonzeros per row:", (learned > 0).sum(axis=1)[:3])

## Q6(a) Results table and prediction plots

In [ ]:
table = plots.results_table(RESULTS_DIR)
print(table.to_string(index=False))
print(table.to_latex(index=False))

In [ ]:
test_timestamps = splits["test"].timestamps
for position, sensor in enumerate(config.prediction_sensors):
    plots.plot_pred_vs_actual(RESULTS_DIR, ["temporal", "fixed", "adaptive", "hybrid"],
                              sensor_position=position, horizon_step=0,
                              test_start="2021-04-05", timestamps_test=test_timestamps,
                              save_path=f"{FIGURE_DIR}/q6a_pred_vs_actual_sensor{sensor}.pdf")

## Q6(b) Graph structure analysis

In [ ]:
for k in (10, 20):
    overlap = plots.edge_overlap_with_fixed(learned, adjacency, k=k)
    print(f"top-{k} learned edges present in road graph: {overlap:.1%}")

In [ ]:
fig, learned_km, road_km = plots.plot_edge_distance_comparison(
    learned, adjacency, meta_sd, k=10, save_path=f"{FIGURE_DIR}/q6b_edge_distances.pdf")
print(f"median distance: learned {np.median(learned_km):.1f} km vs road-graph {np.median(road_km):.1f} km")
print(f"learned edges farther than 10 km: {(learned_km > 10).mean():.1%}")

## Q7 Reproducibility record

In [ ]:
import os
print("Python:", platform.python_version(), "| torch:", torch.__version__, "| numpy:", np.__version__)
print("Platform:", platform.platform(), "| CPU:", platform.processor(), "| cores:", os.cpu_count())
print("CUDA available:", torch.cuda.is_available())
print("seed:", config.seed, "| hidden_dim:", config.hidden_dim, "| lr:", config.learning_rate,
      "| batch:", config.batch_size, "| max_epochs:", config.max_epochs, "| patience:", config.patience)
for variant, result in results.items():
    print(variant, f"train {result['training_seconds']:.0f}s over {result['epochs_run']} epochs "
          f"({result['seconds_per_epoch']:.1f}s/epoch), best epoch {result['best_epoch']}")